# 🔄 Customer Churn Prediction — End-to-End ML Pipeline

**Author:** Deepanshu  
**GitHub:** [deepanshu0110](https://github.com/deepanshu0110)  
**Dataset:** [Telco Customer Churn (Kaggle)](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

---

## 📌 Business Problem

A telecom company is experiencing **high customer attrition**. Each churned customer costs the company approximately **5–7x more to replace** than to retain. The business needs a predictive model to:

1. **Identify customers at high risk of churning** before they leave
2. **Understand the key drivers** behind churn decisions
3. **Enable the retention team** to launch targeted interventions (discounts, plan upgrades, support calls)

## 📊 Project Pipeline

| Stage | What We Do |
|-------|-----------|
| 1. Data Loading & Inspection | Load data, check types, missing values, shape |
| 2. Exploratory Data Analysis | Univariate & bivariate analysis, churn distribution |
| 3. Data Preprocessing | Encoding, scaling, handling imbalance |
| 4. Feature Engineering | Create business-meaningful features |
| 5. Model Building | Logistic Regression, Random Forest, XGBoost |
| 6. Model Evaluation | Confusion Matrix, ROC-AUC, Precision-Recall |
| 7. Hyperparameter Tuning | GridSearchCV on best model |
| 8. Feature Importance & SHAP | Explainability for business stakeholders |
| 9. Business Recommendations | Actionable retention strategies |


## 1. Setup & Data Loading

In [ ]:
# ============================================================
# IMPORTS — every library we'll need for the full pipeline
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils import class_weight

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

# Evaluation
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, f1_score, accuracy_score,
    average_precision_score
)

# Styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("✅ All libraries loaded successfully.")


In [ ]:
# ============================================================
# LOAD DATA — Telco Customer Churn dataset
# ============================================================
# Option 1: Load from local CSV
# df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Option 2: Load from URL (Kaggle mirror)
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print(f"Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
df.head()


## 2. Data Inspection

Before any modeling, we need to understand the data's structure, types, and quality issues.


In [ ]:
# ============================================================
# DATA TYPES & STRUCTURE
# ============================================================
print("=" * 60)
print("COLUMN TYPES & NON-NULL COUNTS")
print("=" * 60)
df.info()


In [ ]:
# ============================================================
# STATISTICAL SUMMARY — separate numeric and categorical
# ============================================================
print("\n📊 Numeric Columns Summary:")
print(df.describe().round(2))

print("\n📊 Categorical Columns Summary:")
print(df.describe(include='object'))


In [ ]:
# ============================================================
# MISSING VALUES — critical check before preprocessing
# ============================================================
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if len(missing_df) > 0:
    print("⚠️ Columns with missing values:")
    print(missing_df)
else:
    print("✅ No null values found.")

# Check for hidden missing values — TotalCharges often has spaces
print(f"\n🔍 TotalCharges dtype: {df['TotalCharges'].dtype}")
print(f"   Empty strings in TotalCharges: {(df['TotalCharges'] == ' ').sum()}")


In [ ]:
# ============================================================
# FIX TotalCharges — convert from object to float
# ============================================================
# TotalCharges has ' ' (space) for new customers with 0 tenure
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill NaN with 0 (these are new customers with tenure = 0)
print(f"NaN after conversion: {df['TotalCharges'].isna().sum()}")
df['TotalCharges'].fillna(0, inplace=True)

print("✅ TotalCharges cleaned and converted to float64.")
print(f"   dtype now: {df['TotalCharges'].dtype}")


In [ ]:
# ============================================================
# DUPLICATE CHECK
# ============================================================
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")

# Drop customerID — it's an identifier, not a feature
df.drop('customerID', axis=1, inplace=True)
print(f"\n✅ Dropped 'customerID'. New shape: {df.shape}")


## 3. Exploratory Data Analysis (EDA)

### 3.1 Target Variable Distribution
Understanding the class balance is critical — imbalanced targets can mislead accuracy metrics.


In [ ]:
# ============================================================
# TARGET DISTRIBUTION — Churn is binary (Yes/No)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
churn_counts = df['Churn'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(churn_counts.index, churn_counts.values, color=colors, edgecolor='black')
axes[0].set_title('Churn Distribution (Count)', fontweight='bold')
axes[0].set_ylabel('Number of Customers')

# Add count labels
for i, (idx, val) in enumerate(zip(churn_counts.index, churn_counts.values)):
    axes[0].text(i, val + 50, f'{val}', ha='center', fontweight='bold', fontsize=13)

# Percentage plot
churn_pct = (churn_counts / len(df) * 100).round(1)
axes[1].pie(churn_pct, labels=[f'No ({churn_pct["No"]}%)', f'Yes ({churn_pct["Yes"]}%)'],
            colors=colors, autopct='%1.1f%%', startangle=90,
            explode=(0, 0.05), shadow=True, textprops={'fontsize': 13})
axes[1].set_title('Churn Distribution (%)', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n📊 Churn Rate: {churn_pct['Yes']}%")
print(f"   Class Ratio (No:Yes): {churn_counts['No']}:{churn_counts['Yes']} ≈ {churn_counts['No']/churn_counts['Yes']:.1f}:1")
print("\n⚠️ Moderate class imbalance — we'll handle this during modeling.")


### 3.2 Numeric Features Analysis

In [ ]:
# ============================================================
# NUMERIC FEATURES — Distribution by Churn status
# ============================================================
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(numeric_cols):
    # KDE plot split by churn
    for label, color in zip(['No', 'Yes'], ['#2ecc71', '#e74c3c']):
        subset = df[df['Churn'] == label][col]
        axes[i].hist(subset, bins=30, alpha=0.5, label=f'Churn={label}',
                     color=color, edgecolor='black')
    
    axes[i].set_title(f'{col} by Churn', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].legend()

plt.tight_layout()
plt.show()

# Key stats
print("\n📊 Median Values by Churn Status:")
for col in numeric_cols:
    no_med = df[df['Churn']=='No'][col].median()
    yes_med = df[df['Churn']=='Yes'][col].median()
    print(f"   {col:20s} — Stayed: {no_med:>8.1f} | Churned: {yes_med:>8.1f}")


### 3.3 Categorical Features vs Churn

In [ ]:
# ============================================================
# CATEGORICAL FEATURES — Churn rate by category
# ============================================================
cat_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
            'PhoneService', 'MultipleLines', 'InternetService',
            'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies',
            'Contract', 'PaperlessBilling', 'PaymentMethod']

fig, axes = plt.subplots(4, 4, figsize=(22, 20))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    # Calculate churn rate per category
    ct = pd.crosstab(df[col], df['Churn'], normalize='index') * 100
    ct['Yes'].plot(kind='bar', ax=axes[i], color='#e74c3c', edgecolor='black', alpha=0.8)
    axes[i].set_title(f'Churn % by {col}', fontweight='bold', fontsize=11)
    axes[i].set_ylabel('Churn Rate (%)')
    axes[i].set_ylim(0, 70)
    axes[i].axhline(y=26.5, color='black', linestyle='--', alpha=0.5, label='Overall Avg')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


### 3.4 Correlation Heatmap

In [ ]:
# ============================================================
# CORRELATION MATRIX — numeric features + encoded target
# ============================================================
df_corr = df.copy()
df_corr['Churn_Binary'] = (df_corr['Churn'] == 'Yes').astype(int)

corr_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn_Binary']
corr_matrix = df_corr[corr_cols].corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, mask=mask, square=True, linewidths=1)
plt.title('Correlation Matrix', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

print("\n🔑 Key Correlations with Churn:")
print(f"   tenure ↔ Churn:         {corr_matrix.loc['tenure', 'Churn_Binary']:.3f} (longer tenure → less churn)")
print(f"   MonthlyCharges ↔ Churn: {corr_matrix.loc['MonthlyCharges', 'Churn_Binary']:.3f} (higher bills → more churn)")
print(f"   tenure ↔ TotalCharges:  {corr_matrix.loc['tenure', 'TotalCharges']:.3f} (expected strong positive)")


### 3.5 Key EDA Insights

| Finding | Business Implication |
|---------|---------------------|
| ~26.5% churn rate | Moderate imbalance — need careful metric selection |
| Month-to-month contracts churn ~43% | Push annual contracts with incentives |
| Fiber optic customers churn more | Possible service quality or pricing issue |
| No online security → higher churn | Bundle security as a retention lever |
| Electronic check payers churn most | Payment friction indicator |
| Low tenure + high monthly charges = risk | New high-value customers need early attention |


## 4. Data Preprocessing

In [ ]:
# ============================================================
# ENCODE TARGET VARIABLE
# ============================================================
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print(f"Target encoded: {df['Churn'].value_counts().to_dict()}")


In [ ]:
# ============================================================
# FEATURE ENGINEERING — create business-relevant features
# ============================================================

# 1. Average monthly spend (TotalCharges / tenure)
#    Handles tenure=0 edge case
df['AvgMonthlySpend'] = np.where(
    df['tenure'] > 0,
    df['TotalCharges'] / df['tenure'],
    df['MonthlyCharges']
)

# 2. Tenure buckets — business lifecycle stages
df['TenureBucket'] = pd.cut(
    df['tenure'],
    bins=[0, 6, 12, 24, 48, 72],
    labels=['0-6mo', '6-12mo', '1-2yr', '2-4yr', '4-6yr']
)

# 3. Number of services subscribed (measures engagement depth)
service_cols = ['PhoneService', 'MultipleLines', 'InternetService',
                'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

# Count 'Yes' across service columns
df['NumServices'] = df[service_cols].apply(
    lambda row: sum(1 for val in row if val == 'Yes' or val == 'Fiber optic' or val == 'DSL'),
    axis=1
)

# 4. Has any protection service (security/backup/device/tech)
protection_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
df['HasProtection'] = df[protection_cols].apply(
    lambda row: 1 if any(val == 'Yes' for val in row) else 0,
    axis=1
)

# 5. Monthly charges to tenure ratio (high spend + low tenure = risk)
df['ChargePerTenure'] = df['MonthlyCharges'] / (df['tenure'] + 1)

print("✅ Feature Engineering Complete. New features:")
print(f"   AvgMonthlySpend  — mean: {df['AvgMonthlySpend'].mean():.2f}")
print(f"   NumServices      — mean: {df['NumServices'].mean():.2f}")
print(f"   HasProtection    — % with protection: {df['HasProtection'].mean()*100:.1f}%")
print(f"   ChargePerTenure  — mean: {df['ChargePerTenure'].mean():.2f}")
print(f"\n   TenureBucket distribution:")
print(df['TenureBucket'].value_counts().sort_index().to_string())


In [ ]:
# ============================================================
# ENCODE CATEGORICAL VARIABLES
# ============================================================

# Identify all object-type columns (excluding already-processed ones)
cat_features = df.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns to encode ({len(cat_features)}):")
print(cat_features)

# Label Encoding for binary features, One-Hot for multi-class
binary_cols = [col for col in cat_features if df[col].nunique() == 2]
multi_cols = [col for col in cat_features if df[col].nunique() > 2]

print(f"\n   Binary columns ({len(binary_cols)}): {binary_cols}")
print(f"   Multi-class columns ({len(multi_cols)}): {multi_cols}")

# Encode binary columns with LabelEncoder
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

# One-Hot Encode multi-class columns
df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

# Also encode TenureBucket (it's categorical)
df = pd.get_dummies(df, columns=['TenureBucket'], drop_first=True)

print(f"\n✅ Encoding complete. New shape: {df.shape}")


In [ ]:
# ============================================================
# TRAIN-TEST SPLIT
# ============================================================
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples ({y_train.mean()*100:.1f}% churn)")
print(f"Test set:     {X_test.shape[0]} samples ({y_test.mean()*100:.1f}% churn)")
print(f"\n✅ Stratified split ensures equal churn proportions in both sets.")


In [ ]:
# ============================================================
# SCALE NUMERIC FEATURES — StandardScaler for gradient-based models
# ============================================================
numeric_to_scale = ['tenure', 'MonthlyCharges', 'TotalCharges',
                    'AvgMonthlySpend', 'NumServices', 'ChargePerTenure']

scaler = StandardScaler()
X_train[numeric_to_scale] = scaler.fit_transform(X_train[numeric_to_scale])
X_test[numeric_to_scale] = scaler.transform(X_test[numeric_to_scale])

print("✅ Numeric features scaled (StandardScaler).")
print("   fit on train → transform on test (no data leakage).")


## 5. Model Building & Comparison

We'll train multiple models and compare them on **ROC-AUC** (not just accuracy) because:
- Accuracy is misleading with imbalanced data
- ROC-AUC measures ranking ability — "how well does the model separate churners from non-churners?"
- Business cares about **catching churners** (recall) while minimizing false alarms (precision)


In [ ]:
# ============================================================
# MODEL TRAINING — 4 classifiers with class_weight='balanced'
# ============================================================

# Compute class weights for cost-sensitive learning
cw = class_weight.compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weights = {0: cw[0], 1: cw[1]}
print(f"Class weights: {class_weights}")
print(f"   → Churners get {cw[1]/cw[0]:.2f}x more weight in the loss function\n")

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=42
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=5, class_weight='balanced', random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10, class_weight='balanced',
        random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.1,
        random_state=42
    )
}

# Train all models and collect results
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
    
    results[name] = {
        'Accuracy': acc, 'F1-Score': f1, 'ROC-AUC': roc,
        'y_pred': y_pred, 'y_prob': y_prob
    }
    print(f"{name:25s} | Acc: {acc:.4f} | F1: {f1:.4f} | ROC-AUC: {roc:.4f}")

print("\n✅ All models trained and evaluated.")


### 5.1 ROC Curves — Visual Model Comparison

In [ ]:
# ============================================================
# ROC CURVES — all models on one plot
# ============================================================
fig, ax = plt.subplots(figsize=(10, 7))

colors = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c']
for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={res['ROC-AUC']:.3f})",
            color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=13)
ax.set_title('ROC Curves — Model Comparison', fontweight='bold', fontsize=15)
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 5.2 Confusion Matrices

In [ ]:
# ============================================================
# CONFUSION MATRICES — all 4 models
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for i, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['No Churn', 'Churn'],
                yticklabels=['No Churn', 'Churn'])
    axes[i].set_title(f'{name}', fontweight='bold', fontsize=11)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.suptitle('Confusion Matrices — All Models', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


### 5.3 Classification Reports

In [ ]:
# ============================================================
# DETAILED CLASSIFICATION REPORTS
# ============================================================
for name, res in results.items():
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(classification_report(y_test, res['y_pred'],
          target_names=['Not Churned', 'Churned']))


## 6. Hyperparameter Tuning — Gradient Boosting

Gradient Boosting typically performs best on structured/tabular data. Let's tune it with GridSearchCV.


In [ ]:
# ============================================================
# GRIDSEARCHCV — Gradient Boosting hyperparameters
# ============================================================
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.05, 0.1, 0.15],
    'subsample': [0.8, 1.0]
}

gb_tuned = GradientBoostingClassifier(random_state=42)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    gb_tuned, param_grid, cv=cv,
    scoring='roc_auc', n_jobs=-1, verbose=0
)

print("🔍 Running GridSearchCV (this may take 1-2 minutes)...")
grid_search.fit(X_train, y_train)

print(f"\n✅ Best Parameters: {grid_search.best_params_}")
print(f"   Best CV ROC-AUC: {grid_search.best_score_:.4f}")

# Evaluate tuned model on test set
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)
y_prob_tuned = best_model.predict_proba(X_test)[:, 1]

print(f"\n📊 Tuned Model — Test Set Performance:")
print(f"   Accuracy:  {accuracy_score(y_test, y_pred_tuned):.4f}")
print(f"   F1-Score:  {f1_score(y_test, y_pred_tuned):.4f}")
print(f"   ROC-AUC:   {roc_auc_score(y_test, y_prob_tuned):.4f}")


In [ ]:
# ============================================================
# TUNED MODEL — CLASSIFICATION REPORT
# ============================================================
print("\n" + "="*60)
print("  TUNED GRADIENT BOOSTING — Final Classification Report")
print("="*60)
print(classification_report(y_test, y_pred_tuned,
      target_names=['Not Churned', 'Churned']))

# Confusion Matrix
cm_tuned = confusion_matrix(y_test, y_pred_tuned)
plt.figure(figsize=(7, 5))
sns.heatmap(cm_tuned, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=['Not Churned', 'Churned'],
            yticklabels=['Not Churned', 'Churned'])
plt.title('Tuned Gradient Boosting — Confusion Matrix', fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()


## 7. Feature Importance & Model Explainability

Understanding *why* the model predicts churn is as important as the prediction itself. Business stakeholders won't act on a black box.


In [ ]:
# ============================================================
# FEATURE IMPORTANCE — Gradient Boosting (built-in)
# ============================================================
feature_imp = pd.Series(
    best_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=True)

# Top 15 features
top_n = 15
fig, ax = plt.subplots(figsize=(10, 8))
feature_imp.tail(top_n).plot(kind='barh', ax=ax, color='#e74c3c', edgecolor='black')
ax.set_title(f'Top {top_n} Feature Importances — Tuned Gradient Boosting',
             fontweight='bold', fontsize=14)
ax.set_xlabel('Feature Importance (Gini)', fontsize=12)
plt.tight_layout()
plt.show()

print("\n🔑 Top 5 Features Driving Churn:")
for feat, imp in feature_imp.tail(5).items():
    print(f"   {feat:30s} → importance: {imp:.4f}")


In [ ]:
# ============================================================
# PRECISION-RECALL CURVE — critical for imbalanced problems
# ============================================================
precision, recall, thresholds = precision_recall_curve(y_test, y_prob_tuned)
ap = average_precision_score(y_test, y_prob_tuned)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# PR Curve
axes[0].plot(recall, precision, color='#e74c3c', linewidth=2)
axes[0].fill_between(recall, precision, alpha=0.1, color='#e74c3c')
axes[0].set_xlabel('Recall (Sensitivity)', fontsize=13)
axes[0].set_ylabel('Precision', fontsize=13)
axes[0].set_title(f'Precision-Recall Curve (AP={ap:.3f})', fontweight='bold', fontsize=14)
axes[0].grid(True, alpha=0.3)

# Threshold optimization — F1 vs Threshold
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-8)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

axes[1].plot(thresholds, precision[:-1], label='Precision', color='#3498db', linewidth=2)
axes[1].plot(thresholds, recall[:-1], label='Recall', color='#e74c3c', linewidth=2)
axes[1].plot(thresholds, f1_scores, label='F1-Score', color='#2ecc71', linewidth=2)
axes[1].axvline(x=optimal_threshold, color='black', linestyle='--', alpha=0.7,
                label=f'Optimal Threshold={optimal_threshold:.3f}')
axes[1].set_xlabel('Threshold', fontsize=13)
axes[1].set_ylabel('Score', fontsize=13)
axes[1].set_title('Precision / Recall / F1 vs Threshold', fontweight='bold', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Optimal Threshold: {optimal_threshold:.3f}")
print(f"   At this threshold → F1: {f1_scores[optimal_idx]:.3f}")


## 8. Business Recommendations & Deployment Strategy

### 🎯 Actionable Retention Strategies

Based on our analysis, here are prioritized recommendations:

| Priority | Action | Target Segment | Expected Impact |
|----------|--------|----------------|-----------------|
| 🔴 HIGH | Offer 1-year contract discounts | Month-to-month customers with tenure < 12 months | Reduce churn by ~15-20% |
| 🔴 HIGH | Bundle online security + tech support | Fiber optic users without protection services | Reduce churn by ~10-15% |
| 🟡 MEDIUM | Switch to auto-pay incentives | Electronic check payers | Reduce payment friction churn |
| 🟡 MEDIUM | Early engagement program | New customers (tenure 0-6 months) with high monthly charges | Reduce early-stage churn |
| 🟢 LOW | Loyalty rewards program | Customers with tenure > 2 years | Strengthen retention of stable base |

### 📊 Model Deployment Recommendations

1. **Scoring frequency:** Run the model monthly on the active customer base
2. **Threshold:** Use the optimized threshold for production scoring
3. **Integration:** Feed churn scores into the CRM for retention team prioritization
4. **Monitoring:** Track model drift quarterly — retrain when ROC-AUC drops below 0.78
5. **A/B testing:** Validate retention campaigns against a control group

### 💰 Estimated Business Impact

Assuming:
- 7,000 customers, 26.5% churn rate → ~1,855 churners/year
- Average revenue per customer = $65/month = $780/year
- Model catches 75% of churners, retention campaign saves 30% of those contacted

**Estimated revenue saved = 1,855 × 0.75 × 0.30 × $780 ≈ $325,000/year**


## 9. Save Model for Deployment

In [ ]:
# ============================================================
# SAVE MODEL & PREPROCESSING ARTIFACTS
# ============================================================
import pickle

# Save the tuned model
with open('churn_model_gb_tuned.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Save the scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("✅ Artifacts saved:")
print("   churn_model_gb_tuned.pkl — Tuned Gradient Boosting classifier")
print("   scaler.pkl               — StandardScaler (fitted on training data)")
print("\n📌 Use these for Streamlit/Flask deployment or batch scoring.")


In [ ]:
# ============================================================
# QUICK PREDICTION DEMO — predict on a single customer
# ============================================================
print("🔮 Prediction Demo — Random Test Customer\n")

sample_idx = X_test.index[0]
sample = X_test.iloc[[0]]
actual = y_test.iloc[0]
prob = best_model.predict_proba(sample)[0][1]
pred = 'CHURN' if prob >= optimal_threshold else 'NO CHURN'

print(f"   Churn Probability: {prob:.2%}")
print(f"   Prediction (threshold={optimal_threshold:.3f}): {pred}")
print(f"   Actual Label: {'CHURN' if actual == 1 else 'NO CHURN'}")


## 10. Key Takeaways

1. **Gradient Boosting** outperformed other models on ROC-AUC — the right metric for ranking churn risk
2. **Contract type, tenure, and monthly charges** are the strongest churn predictors
3. **Class imbalance** was handled via `class_weight='balanced'` — not SMOTE (avoids synthetic noise)
4. **Threshold tuning** on the PR curve gave a better precision-recall trade-off than default 0.5
5. **Feature engineering** (NumServices, HasProtection, ChargePerTenure) added predictive signal beyond raw features

### 🎤 Interview-Ready Insights

- *"Why not accuracy?"* → With 26.5% churn, a model predicting all 'No Churn' gets 73.5% accuracy but catches zero churners. ROC-AUC and F1 are the right metrics.
- *"Why Gradient Boosting over Random Forest?"* → GB optimizes sequentially (boosting corrects errors), while RF averages independently. GB typically wins on structured tabular data.
- *"How did you handle imbalance?"* → `class_weight='balanced'` adjusts the loss function. More robust than SMOTE for this dataset size.
- *"What's the business ROI?"* → Estimated ~$325K/year in retained revenue at modest assumptions.

---

**📂 Repository:** [github.com/deepanshu0110/customer-churn-prediction](https://github.com/deepanshu0110/customer-churn-prediction)  
**📧 Contact:** Deepanshu — Freelance Data Analyst & Data Scientist
